In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
parameters = json.loads(Path('settings.json').read_text())
ticker = parameters['ticker']
market_map = {'SP500': '^GSPC', 'FTSE': 'FTSE100', 'DAX': 'GDAXI', 'CAC': 'CACT', 'STOXX': 'STOXX', 'CBU0': 'CBU0.L'}
market = next(name for name, value in market_map.items() if value == ticker)
train_benchmarks = False
kind = 'NEWS'


In [ ]:
if train_benchmarks:
    from pathlib import Path
    import contextlib
    import hashlib
    import io
    import json
    import os
    import random
    import time
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    os.environ['OMP_NUM_THREADS'] = '2'
    os.environ['OPENBLAS_NUM_THREADS'] = '2'
    import joblib
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score, precision_recall_curve
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dropout, Dense
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


    from IPython.display import display


In [ ]:
if train_benchmarks:
    root = Path('.').resolve()
    parameters = json.loads(Path('settings.json').read_text())
    ticker = parameters['ticker']
    settings = json.loads(Path('markets.json').read_text())[ticker]
    output_root = root / 'Results' / ticker / parameters['run_id']
    kind = 'NEWS'
    Market = market
    market_map = {'SP500': '^GSPC', 'FTSE': 'FTSE100', 'DAX': 'GDAXI', 'CAC': 'CACT', 'STOXX': 'STOXX', 'CBU0': 'CBU0.L'}
    main_path = root / 'Benchmark/Official/main_results.csv'
    main = pd.read_csv(main_path, parse_dates=['target_date', 'information_cutoff'])
    own = main[main.market.eq(market_map[market])].sort_values('target_date')
    bench = root / 'Data/Benchmark'
    prices = pd.read_csv(bench / f'OHLCV/{market}_df_OHLCV.csv', parse_dates=['Date']).set_index('Date').sort_index()
    next_dates = pd.Series(prices.index, index=prices.index).shift(-1)
    dest = output_root / 'LSTM' / f'{market}_{kind}'
    dest.mkdir(parents=True, exist_ok=False)
    np.random.seed(42)
    random.seed(42)
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(1)


In [ ]:
if train_benchmarks:
    OHLCV_df_list = []
    Market_list = ['SP500', 'STOXX', 'CAC', 'DAX', 'FTSE', 'CBU0']
    for iter_Market in Market_list:
        if iter_Market != Market:
            df = pd.read_csv(f'Data/Benchmark/OHLCV/{iter_Market}_df_OHLCV.csv')
            df = df.rename(columns={col: f'{col}_{iter_Market}' for col in df.columns if col != 'Date'})
            OHLCV_df_list.append(df)
    OHLCV_df = OHLCV_df_list[0]
    for df in OHLCV_df_list[1:]:
        OHLCV_df = OHLCV_df.merge(df, on='Date', how='outer')
    OHLCV_df = OHLCV_df.sort_values('Date').reset_index(drop=True)


In [ ]:
if train_benchmarks:
    crash_labels = pd.read_csv(f'Data/Benchmark/Indicators_crash/{Market}_df_Crash_Label.csv')
    crash_labels = crash_labels.sort_values('date').reset_index(drop=True).ffill().dropna().reset_index(drop=True)
    indicators = pd.read_csv(f'Data/Benchmark/Indicators_crash/{Market}_df_Indicators.csv')
    print(f'Missing values in crash labels: {crash_labels.isna().sum().sum()}')
    print(f'Missing values in indicators: {indicators.isna().sum().sum()}')
    indicators = indicators.dropna()
    print(indicators.shape)
    indicators = indicators.merge(OHLCV_df, on='Date', how='left')
    cols = [col for col in indicators.columns if 'Date_' not in col]
    indicators = indicators[cols]
    indicators = indicators.dropna(axis=1)
    print(indicators.shape)


In [ ]:
if train_benchmarks:
    primary_price_path = root / f'Data/{market_map[market]}/Market/df_OHLCV.csv'
    primary_prices = pd.read_csv(primary_price_path, parse_dates=['Date']).set_index('Date').sort_index()
    primary_returns = np.log(primary_prices.Close).diff()
    primary_threshold = primary_returns.rolling(252).quantile(.05).shift(1)
    primary_future = primary_returns.shift(-1)
    primary_y = (primary_future <= primary_threshold).astype(float).where(primary_threshold.notna() & primary_future.notna())
    assert np.array_equal(primary_y.reindex(own.information_cutoff).to_numpy(), own.actual_label.to_numpy())
    primary_labels = pd.DataFrame({'date': primary_prices.index, 'crash_label': primary_y.to_numpy(), 'min_future_logret': primary_future.to_numpy(), 'crash_threshold': primary_threshold.to_numpy()})
    primary_labels['date'] = primary_labels['date'].dt.strftime('%Y-%m-%d')
    crash_labels = primary_labels.sort_values('date').reset_index(drop=True).ffill().dropna().reset_index(drop=True)


In [ ]:
if train_benchmarks:
    crash_labels['date'] = pd.to_datetime(crash_labels['date'])
    indicators['Date'] = pd.to_datetime(indicators['Date'])
    date_cols = ['date', 'Date']
    label_cols = ['crash_label', 'min_future_logret', 'crash_threshold']


In [ ]:
if train_benchmarks:
    from sklearn.decomposition import PCA
    news_embeddings_df = pd.read_csv(f'Data/Benchmark/{Market}_daily_news_embeddings.csv', parse_dates=['date'])
    embedding_cols = [col for col in news_embeddings_df.columns if col != 'date']
    components = 10
    X = news_embeddings_df[embedding_cols].values
    pca = PCA(n_components=components, random_state=42)
    X_pca = pca.fit_transform(X)
    pca_cols = [f'news_pca_{i}' for i in range(components)]
    news_pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=news_embeddings_df.index)
    news_pca_df['date'] = news_embeddings_df['date'].values
    print('Original shape:', X.shape)
    print('PCA shape:', X_pca.shape)
    print('Explained variance:', pca.explained_variance_ratio_.sum())


In [ ]:
if train_benchmarks:
    print('Merging news embeddings with indicators...')
    indicators['date'] = pd.to_datetime(indicators['Date'])
    merged_with_news = indicators.merge(news_pca_df, on='date', how='left')
    embedding_cols_list = [col for col in merged_with_news.columns if col.startswith('news_pca_')]
    merged_with_news[embedding_cols_list] = merged_with_news[embedding_cols_list].fillna(0)
    print(f'Merged dataset shape: {merged_with_news.shape}')
    print(f'Original indicators columns: {len(indicators.columns)}')
    print(f'New columns added: {len(embedding_cols_list)}')
    crash_labels['date'] = pd.to_datetime(crash_labels['date'])
    merged_df_with_news = crash_labels.merge(merged_with_news, on='date', how='inner')
    print(f'Final merged dataset shape: {merged_df_with_news.shape}')
    merged_df_with_news = merged_df_with_news.sort_values('date').reset_index(drop=True)
    indicator_cols_new = [col for col in merged_df_with_news.columns if col not in ['date', 'Date', 'crash_label', 'min_future_logret', 'crash_threshold']]
    print(f'\nTotal features for LSTM: {len(indicator_cols_new)}')
    print(f"  - Technical indicators: {len([c for c in indicator_cols_new if not c.startswith('news_pca_')])}")
    print(f"  - News embeddings PCA: {len([c for c in indicator_cols_new if c.startswith('news_pca_')])}")


In [ ]:
if train_benchmarks:
    X_indicators_with_news = merged_df_with_news[indicator_cols_new].values
    y_crash_with_news = merged_df_with_news['crash_label'].values.astype(np.int32)
    y_crash_with_news = pd.Series(y_crash_with_news).ffill().values
    dates_seq_with_news = merged_df_with_news['date'].values
    print(f'Total observations: {len(X_indicators_with_news)}')
    print(f'Number of features: {X_indicators_with_news.shape[1]}')
    print('\nOriginal class distribution:')
    print(np.bincount(y_crash_with_news))
    print(f'Overall anomaly rate: {y_crash_with_news.mean():.4%}')


In [ ]:
if train_benchmarks:
    seq_length = 21
    X_seq = []
    y_seq = []
    dates_seq = []
    for i in range(len(X_indicators_with_news) - seq_length):
        X_seq.append(X_indicators_with_news[i:i + seq_length])
        y_seq.append(y_crash_with_news[i + seq_length])
        dates_seq.append(dates_seq_with_news[i + seq_length])
    X_seq = np.asarray(X_seq)
    y_seq = np.asarray(y_seq).astype(np.int32)
    dates_seq = pd.to_datetime(np.asarray(dates_seq))
    print(f'\nSequence shape: {X_seq.shape}')
    print(f'Labels shape: {y_seq.shape}')
    print('\nOverall label distribution:')
    print(pd.Series(y_seq).value_counts())
    print(f'\nOverall anomaly rate: {y_seq.mean():.4%}')


In [ ]:
if train_benchmarks:
    TEST_CUTOFF = pd.to_datetime('2025-03-01')
    N_SPLITS = 5
    INITIAL_TRAIN_FRACTION = 0.6
    PURGE_GAP = 21
    EPOCHS = 100
    BATCH_SIZE = 32
    LSTM_UNITS = 40
    DENSE_UNITS = 20
    DROPOUT_1 = 0.3
    DROPOUT_2 = 0.3
    LEARNING_RATE = 0.0005
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = 0.2
    EARLY_STOP_PATIENCE = 50
    LR_PATIENCE = 4
    LR_FACTOR = 0.5
    MIN_LR = 1e-06


In [ ]:
if train_benchmarks:
    def scale_sequences(X, scaler):









        original_shape = X.shape
        X_2d = X.reshape(-1, original_shape[-1])
        X_scaled = scaler.transform(X_2d)
        return X_scaled.reshape(original_shape)

    def calculate_class_weights(y):
        n_normal = np.sum(y == 0)
        n_anomaly = np.sum(y == 1)
        if n_anomaly == 0:
            raise ValueError('Training window contains no anomaly samples.')
        weight_normal = 1.0
        weight_anomaly = n_normal / n_anomaly
        class_weight = {0: weight_normal, 1: max(5.0, weight_anomaly)}
        return (class_weight, n_normal, n_anomaly, weight_anomaly)

    def build_model():
        model = Sequential([LSTM(LSTM_UNITS, return_sequences=False, input_shape=(seq_length, n_features)), Dropout(DROPOUT_1), Dense(DENSE_UNITS, activation='relu'), Dropout(DROPOUT_2), Dense(1, activation='sigmoid')])
        model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=tf.keras.losses.BinaryFocalCrossentropy(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA), metrics=[tf.keras.metrics.AUC(name='roc_auc'), tf.keras.metrics.AUC(name='pr_auc', curve='PR'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
        return model


In [ ]:
if train_benchmarks:
    x, y, label_dates = X_seq, y_seq, pd.DatetimeIndex(dates_seq)
    order = np.argsort(label_dates)
    x, y, label_dates = x[order], y[order], label_dates[order]
    target_dates = pd.DatetimeIndex(next_dates.reindex(label_dates).to_numpy())
    dev = np.flatnonzero(target_dates < own.target_date.min())
    test = np.flatnonzero(target_dates.isin(own.target_date))
    evaluated = own[own.target_date.isin(target_dates[test])].copy()
    assert len(test) == len(own)
    assert np.array_equal(target_dates[test], evaluated.target_date)
    assert np.array_equal(y[test], evaluated.actual_label.to_numpy())
    assert np.array_equal(label_dates[test], evaluated.information_cutoff)
    merged = merged_df if kind == 'IND' else merged_df_with_news
    raw_dates = pd.DatetimeIndex(pd.to_datetime(merged['date']))
    end_positions = raw_dates.get_indexer(label_dates) - 1
    feature_end_dates = raw_dates[end_positions]
    features = indicator_cols if kind == 'IND' else indicator_cols_new
    X_dev, y_dev, dates_dev = x[dev], y[dev], label_dates[dev]
    X_test, y_test, dates_test = x[test], y[test], label_dates[test]
    n_features, n_samples = x.shape[-1], len(dev)
    np.savez_compressed(dest / 'selected_inputs.npz', development_index=dev, test_index=test, label_dates=label_dates.to_numpy(), target_dates=target_dates.to_numpy(), feature_end_dates=feature_end_dates.to_numpy(), labels=y)
    (dest / 'features.json').write_text(json.dumps(features, indent=2) + '\n')
    display(pd.Series({'market': market, 'model': kind, 'development_n': len(dev), 'test_n': len(test), 'features': n_features, 'first_target': target_dates[test].min(), 'last_target': target_dates[test].max()}))


In [ ]:
if train_benchmarks:
    initial_train_size = int(n_samples * INITIAL_TRAIN_FRACTION)
    remaining_samples = n_samples - initial_train_size
    if remaining_samples < N_SPLITS:
        raise ValueError('Not enough samples for the requested number of validation folds.')
    validation_block_size = remaining_samples // N_SPLITS
    walk_forward_splits = []
    for fold in range(N_SPLITS):
        val_start = initial_train_size + fold * validation_block_size
        if fold == N_SPLITS - 1:
            val_end = n_samples
        else:
            val_end = val_start + validation_block_size
        val_indices = np.arange(val_start, val_end)
        train_end = max(0, val_start - PURGE_GAP)
        train_indices = np.arange(0, train_end)
        walk_forward_splits.append((train_indices, val_indices))


In [ ]:
if train_benchmarks:
    if train_benchmarks:
        oof = np.full(len(dev), np.nan)
        test_predictions = []
        for k, (train_idx, val_idx) in enumerate(walk_forward_splits, start=1):
            started = time.time()
            scaler = StandardScaler().fit(X_dev[train_idx].reshape(-1, x.shape[-1]))
            train_x = scale_sequences(X_dev[train_idx], scaler)
            val_x = scale_sequences(X_dev[val_idx], scaler)
            test_x = scale_sequences(X_test, scaler)
            train_y, val_y = y_dev[train_idx], y_dev[val_idx]
            weights = calculate_class_weights(train_y)[0]
            tf.keras.backend.clear_session()
            model = build_model()
            callbacks = [EarlyStopping(monitor='val_roc_auc', mode='max', patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=0), ReduceLROnPlateau(monitor='val_roc_auc', mode='max', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR, verbose=0)]
            print(f'START {market} {kind} fold={k} train={len(train_idx)} val={len(val_idx)} features={x.shape[-1]}', flush=True)
            history = model.fit(train_x, train_y, validation_data=(val_x,val_y), epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=weights, callbacks=callbacks, verbose=0)
            oof[val_idx] = model.predict(val_x, verbose=0).ravel()
            test_predictions.append(model.predict(test_x, verbose=0).ravel())
            model.save(dest/f'fold_{k}.keras')
            joblib.dump(scaler, dest/f'fold_{k}_scaler.joblib')
            pd.DataFrame(history.history).to_csv(dest/f'fold_{k}_history.csv', index=False)
            np.savez_compressed(dest/f'fold_{k}_predictions.npz', validation=oof[val_idx], test=test_predictions[-1], train_index=train_idx, validation_index=val_idx)
            (dest/f'fold_{k}.json').write_text(json.dumps({'epochs':len(history.history['loss']), 'seconds': time.time()-started, 'class_weights': weights, 'last_train_label_date': str(dates_dev[train_idx[-1]].date()), 'first_validation_label_date': str(dates_dev[val_idx[0]].date())}, indent=2)+'\n')
            print(f'DONE {market} {kind} fold={k} epochs={len(history.history["loss"])} seconds={time.time()-started:.1f}', flush=True)
        valid = np.isfinite(oof)
        precision, recall, thresholds = precision_recall_curve(y_dev[valid], oof[valid])
        f1 = 2*precision[:-1]*recall[:-1]/(precision[:-1]+recall[:-1]+1e-8)
        threshold = float(thresholds[np.argmax(f1)])
        probability = np.mean(test_predictions, axis=0)
        prediction = (probability>=threshold).astype(int)
        tn, fp, fn, tp = map(int, confusion_matrix(y[test], prediction, labels=[0,1]).ravel())
        values = {'n':len(test), 'events':int(y[test].sum()), 'roc_auc':roc_auc_score(y[test],probability), 'ap':average_precision_score(y[test],probability), 'accuracy':accuracy_score(y[test],prediction), 'precision':precision_score(y[test],prediction,zero_division=0), 'recall':recall_score(y[test],prediction,zero_division=0), 'f1':f1_score(y[test],prediction,zero_division=0), 'mcc':matthews_corrcoef(y[test],prediction), 'tp':tp,'fp':fp,'fn':fn,'tn':tn}
        pd.DataFrame({'target_date':target_dates[test], 'original_label_date':label_dates[test], 'feature_end_date':feature_end_dates[test], 'actual':y[test], 'probability':probability, 'prediction':prediction}).to_csv(dest/'test_predictions.csv',index=False)
        pd.DataFrame({'original_label_date':label_dates[dev][valid], 'actual':y[dev][valid], 'probability':oof[valid]}).to_csv(dest/'oof_predictions.csv',index=False)
        if kind=='NEWS':
            joblib.dump(pca,dest/'original_full_period_pca.joblib')
        (dest/'metrics.json').write_text(json.dumps({'market':market,'model':kind,'threshold':threshold,**values},indent=2)+'\n')
        print('RESULT '+json.dumps({'market':market,'model':kind,'threshold':threshold,**values}),flush=True)


In [ ]:
saved = Path('Results') / ticker / parameters['run_id'] / 'LSTM' / f'{market}_{kind}' if train_benchmarks else Path('Benchmark/Official') / f'{market}_{kind}'
display(pd.Series(json.loads((saved / 'metrics.json').read_text())))
display(pd.read_csv(saved / 'test_predictions.csv').head())
